# **Multigrams: Full Workflow**
The multigram workflow mirrors the unigram workflow with one difference: instead of _creating_ a whitelist, you filter the multigram corpus _using_ the whitelist of top-_N_ unigrams. The filter stage also **pivots** the result as it goes: the output database is keyed by year first, so you can learn how many times "nuclear" appeared in 2011 by querying the key `[2011] nuclear`. This is the layout model training reads.

## **Setup**
### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from ngramprep.ngram_acquire import download_and_ingest_to_rocksdb
from lexichron.stages.filter import filter_ngrams
from ngramprep.utilities.peek import db_head, db_peek, db_peek_prefix
from ngramprep.utilities.count_items import count_db_items

### Configure
Here we set basic parameters: the corpus to download, the size of the ngrams to download, and the size of the year bins.

In [ ]:
db_path_stub = '/scratch/edk202/NLP_corpora/Google_Books/'
archive_path_stub = None
release = '20200217'
language = 'eng-us'
ngram_size = 5
bin_size = 1

: 

## **Step 1: Download and Ingest**

Specifying `combined_bigrams_download` will convert compound terms into single, hyphenated tokens. This process is case-sensitive, so we specify all common capitalization patterns.

If you're resuming downloads after an interruption, there may be a lag before you see any output. The RocksDB is being repaired.

In [ ]:
combined_bigrams_download = {
    "human being", "Human being", "Human Being", "human beings", "Human beings", "Human Beings"
}

download_and_ingest_to_rocksdb(
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    archive_path_stub=None,
    ngram_type="tagged",
    random_seed=321,
    overwrite_db=False,
    workers=45,
    write_batch_size=25_000,
    open_type="write:packed24",
    compact_after_ingest=False,
    combined_bigrams=None
)

N-GRAM ACQUISITION PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-03-13 00:54:27

Download Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Ngram repo:           https://books.storage.googleapis.com/?prefix=ngrams/books/20200217/eng-us/5-
DB path:              /scratch/edk202/NLP_corpora/Google_Books/20200217/eng-us/5gram_files/5grams.db
File range:           0 to 11144
Total files:          11145
Files to get:         4547
Skipping:             6598
Download workers:     45
Batch size:           25,000
Ngram size:           5
Ngram type:           tagged
Overwrite DB:         False
DB Profile:           write:packed24

Download Progress
════════════════════════════════════════════════════════════════════════════════════════════════════


Files Processed:   3%|█▌                                                    | 134/4547 [01:15<17:49]

## **Step 2: Filter, Normalize and Pivot**
`filter_ngrams` is the function behind `lexichron filter`. For 2-5-grams it applies the vocabulary from the unigram workflow (`1grams_processed.db/whitelist.txt`, found automatically), lemmatizing each token the same way before the lookup so inflected forms are not lost, replaces out-of-vocabulary tokens with `<UNK>`, drops n-grams with fewer than `min_context_tokens` real tokens, and writes the pivoted (year-first) database `5grams_pivoted.db` in the same pass. Pass `pivot=False` to get an n-gram-keyed `5grams_processed.db` instead.

In [ ]:
# Focal terms to keep even when they are not in the whitelist: pronouns and
# other stopwords the unigram stage dropped. (They are lemmatized before the
# lookup, so inflections need not be listed.)
always_include_tokens = [
    'he', 'she',
    'his', 'her',
    'himself', 'herself',
    'man', 'woman',
    'male', 'female',
    'boy', 'girl',
    'father', 'mother',
    'son', 'daughter',
    'brother', 'sister',
]

filter_ngrams(
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    bin_size=bin_size,
    always_include=always_include_tokens,
    workers=36,
    mode="restart",
);

## **Optional: Inspect Database Files**

### `db_head`: Show first N records

In [ ]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_pivoted.db'

db_head(db, n=5)

### `db_peek`: Show Records starting from a key

In [ ]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_pivoted.db'

# keys are year-first: "[year] ngram"
db_peek(db, start_key="[2011] nuclear", n=5)

### `db_peek_prefix`: Show records matching a prefix

In [ ]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_pivoted.db'

db_peek_prefix(db, prefix="[2011] transplant", n=5)

## **Optional: Count Database Items**

In [ ]:
raw_db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams.db'

raw_count = count_db_items(raw_db)

DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/5grams.db
Progress interval: every 10,000,000 items

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────


In [ ]:
pivoted_db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_pivoted.db'

pivoted_count = count_db_items(pivoted_db, progress_interval=50_000_000)

In [ ]:
pivoted_db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_pivoted.db'

pivoted_count_per_bin = count_db_items(pivoted_db, progress_interval=50_000_000, grouping='year_bin')